# Chapter 3: Unsupervised Learning
## *Introduction to Machine Learning with Python* — Müller & Guido

---
> **Ringkasan Bab:** Bab ini membahas Unsupervised Learning  metode ML yang bekerja tanpa label. Topik utama: **Preprocessing & Scaling**, **Dimensionality Reduction (PCA, NMF, t-SNE)**, dan **Clustering (K-Means, DBSCAN, Agglomerative)**.

## 3.1 Apa itu Unsupervised Learning?

**Unsupervised Learning** = belajar dari data **tanpa label**.

**Dua tujuan utama:**

| Tujuan | Teknik | Contoh |
|---|---|---|
| **Transformasi Data** | PCA, NMF, t-SNE | Kompresi dimensi, visualisasi |
| **Clustering** | K-Means, DBSCAN | Segmentasi pelanggan, deteksi anomali |

**Tantangan:** Tidak ada jawaban "benar" yang bisa digunakan untuk evaluasi → sulit mengetahui apakah hasil baik atau tidak.

## 3.2 Preprocessing & Feature Scaling

Banyak algoritma ML (terutama berbasis jarak) sensitif terhadap **skala fitur**.

| Scaler | Cara Kerja | Kapan Digunakan |
|---|---|---|
| **StandardScaler** | Mean=0, Std=1 | Data berdistribusi normal |
| **MinMaxScaler** | Skala ke range [0,1] | Ingin batas jelas 0–1 |
| **RobustScaler** | Gunakan median & IQR | Data punya banyak outlier |
| **Normalizer** | Panjang vektor = 1 | Data teks/vektor arah |

> ⚠️ **Aturan penting:** Scaler di-fit **hanya** pada training data, lalu di-transform ke training DAN test data. Jangan fit di test data!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=1
)

# Bandingkan 3 scaler
scalers = {
    'Original'      : None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler'  : MinMaxScaler(),
    'RobustScaler'  : RobustScaler(),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, scaler) in zip(axes, scalers.items()):
    if scaler:
        X_plot = scaler.fit_transform(X_train)
    else:
        X_plot = X_train
    ax.scatter(X_plot[:, 0], X_plot[:, 1], c=y_train, cmap='viridis', alpha=0.6, s=20)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Feature 0')
    ax.set_ylabel('Feature 1')

plt.suptitle('Perbandingan Scaler pada Breast Cancer Dataset', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 3.3 Dimensionality Reduction

**Dimensionality Reduction** = mengurangi jumlah fitur sambil mempertahankan informasi penting.

**Manfaat:**
- Visualisasi data berdimensi tinggi
- Mengurangi noise
- Mempercepat training
- Mengatasi *curse of dimensionality*

### 3.3.1 Principal Component Analysis (PCA)

**PCA** mencari arah (komponen utama) dalam data yang **memaksimalkan variansi**.

**Cara kerja:**
1. Pusatkan data (kurangi mean)
2. Hitung *eigenvector* dari matriks kovarians
3. Pilih k eigenvector dengan eigenvalue terbesar → komponen utama
4. Proyeksikan data ke komponen tersebut

**PCA menghasilkan fitur baru (PC1, PC2, ...) yang:**
- Orthogonal (tidak berkorelasi satu sama lain)
- Berurutan: PC1 menangkap variansi terbesar, PC2 terbesar kedua, dst.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Selalu scale dulu sebelum PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cancer.data)

# PCA ke 2 komponen untuk visualisasi
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Shape sebelum PCA:", X_scaled.shape)
print("Shape setelah PCA:", X_pca.shape)
print(f"\nVariansi yang dijelaskan tiap komponen:")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"  Total: {sum(pca.explained_variance_ratio_):.2%}")

# Visualisasi
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[cancer.target==0, 0], X_pca[cancer.target==0, 1],
            label='Malignant', alpha=0.7, c='red', s=30)
plt.scatter(X_pca[cancer.target==1, 0], X_pca[cancer.target==1, 1],
            label='Benign', alpha=0.7, c='blue', s=30)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('PCA — Breast Cancer Dataset (2 Components)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Berapa komponen yang dibutuhkan untuk menjelaskan 95% variansi?
pca_full = PCA()
pca_full.fit(X_scaled)

cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.argmax(cumulative_var >= 0.95) + 1

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(cumulative_var)+1), cumulative_var, 'o-', color='teal')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% variance')
plt.axvline(x=n_95, color='orange', linestyle='--', label=f'{n_95} components')
plt.xlabel('Jumlah Komponen PCA')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA: Jumlah Komponen vs Variansi Kumulatif')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Butuh {n_95} komponen untuk menjelaskan ≥95% variansi")
print(f"(dari total {cancer.data.shape[1]} fitur asli)")

### 3.3.2 Non-Negative Matrix Factorization (NMF)

**NMF** memfaktorkan matriks data menjadi dua matriks non-negatif: $X \approx W \cdot H$

**Perbedaan dengan PCA:**
- Semua nilai **non-negatif** → lebih interpretatif
- Komponen bisa dianggap sebagai "bagian" dari data
- Cocok untuk data yang secara alami non-negatif: gambar, teks, audio

**Penggunaan umum:** topic modeling, pemisahan sinyal audio, analisis ekspresi gen.

In [ ]:
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler

scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(cancer.data)

nmf = NMF(n_components=2, random_state=0)
X_nmf = nmf.fit_transform(X_mm)

plt.figure(figsize=(8, 6))
plt.scatter(X_nmf[cancer.target==0, 0], X_nmf[cancer.target==0, 1],
            label='Malignant', alpha=0.7, c='red', s=30)
plt.scatter(X_nmf[cancer.target==1, 0], X_nmf[cancer.target==1, 1],
            label='Benign', alpha=0.7, c='blue', s=30)
plt.xlabel('NMF Component 1')
plt.ylabel('NMF Component 2')
plt.title('NMF — Breast Cancer Dataset (2 Components)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3.3 t-SNE (t-distributed Stochastic Neighbor Embedding)

**t-SNE** adalah teknik visualisasi yang sangat baik untuk data berdimensi tinggi.

**Cara kerja:**
- Mengukur kesamaan antar titik di dimensi tinggi
- Mempertahankan hubungan "titik yang dekat" saat diproyeksikan ke 2D/3D

**Catatan penting:**
- **Hanya untuk visualisasi**, bukan untuk preprocessing sebelum training
- Tidak menghasilkan transformasi yang bisa diterapkan ke data baru
- Hasil bisa berbeda tiap kali dijalankan (tergantung `random_state`)
- Parameter `perplexity` mempengaruhi struktur cluster yang terlihat

In [ ]:
from sklearn.manifold import TSNE

from sklearn.datasets import load_digits
digits = load_digits()

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(digits.data)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1],
                      c=digits.target, cmap='tab10', alpha=0.7, s=20)
plt.colorbar(scatter, label='Digit')
plt.title('t-SNE Visualization — Digits Dataset (10 classes)')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.tight_layout()
plt.show()
print("Perhatikan: kelas yang sama cenderung berkumpul bersama!")

## 3.4 Clustering

**Clustering** = mengelompokkan data ke dalam kelompok (**cluster**) berdasarkan kemiripan, tanpa label.

### 3.4.1 K-Means Clustering

**K-Means** adalah algoritma clustering paling populer.

**Cara kerja:**
1. Tentukan jumlah cluster **k**
2. Inisialisasi k titik pusat (*centroid*) secara acak
3. Assign setiap titik ke centroid terdekat
4. Update centroid = rata-rata semua titik di cluster tersebut
5. Ulangi langkah 3-4 sampai centroid tidak berubah

**Kelebihan:** Cepat, sederhana, scalable  
**Kekurangan:** Harus tentukan k dulu, asumsi cluster berbentuk bulat, sensitif terhadap outlier

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# Buat dataset sintetis dengan 3 cluster
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, random_state=42)

# Fit K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X_blobs)

# Visualisasi
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c='gray', alpha=0.5, s=30)
plt.title('Data Asli (Tanpa Label)')
plt.xlabel('Feature 0')
plt.ylabel('Feature 1')

plt.subplot(1, 2, 2)
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans.labels_, cmap='viridis', alpha=0.7, s=30)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c='red', marker='X', s=200, zorder=5, label='Centroid')
plt.title('Hasil K-Means Clustering (k=3)')
plt.xlabel('Feature 0')
plt.legend()

plt.suptitle('K-Means Clustering', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Menentukan k optimal: Elbow Method
inertias = []
k_values = range(1, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blobs)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, 'o-', color='steelblue', linewidth=2)
plt.axvline(x=3, color='red', linestyle='--', label='Optimal k=3')
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('Inertia (Within-cluster Sum of Squares)')
plt.title('Elbow Method untuk Menentukan k Optimal')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Pilih k di titik 'siku' (elbow) — penurunan drastis mulai melambat")

### 3.4.2 Agglomerative Clustering (Hierarchical)

**Agglomerative Clustering** membangun hierarki cluster dari bawah ke atas:
1. Mulai: setiap titik adalah cluster sendiri
2. Gabungkan dua cluster yang paling mirip
3. Ulangi sampai semua menjadi satu cluster

**Linkage (cara mengukur jarak antar cluster):**
- `ward`: minimasi variansi dalam cluster (paling sering digunakan)
- `complete`: jarak maksimum antara titik-titik di dua cluster
- `average`: rata-rata jarak antar semua pasang titik
- `single`: jarak minimum antara titik-titik di dua cluster

**Visualisasi:** Dendrogram

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Fit Agglomerative Clustering
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X_blobs)

# Dendrogram
plt.figure(figsize=(10, 5))
linked = linkage(X_blobs[:50], method='ward')  # gunakan subset agar tidak terlalu ramai
dendrogram(linked, leaf_rotation=90, leaf_font_size=8)
plt.title('Dendrogram — Agglomerative Clustering (Ward Linkage)')
plt.xlabel('Sample Index')
plt.ylabel('Jarak')
plt.tight_layout()
plt.show()

### 3.4.3 DBSCAN

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) mengelompokkan titik berdasarkan **kepadatan**.

**Konsep penting:**
- **Core point:** titik dengan ≥ `min_samples` tetangga dalam radius `eps`
- **Border point:** dekat core point tapi bukan core point sendiri
- **Noise point (outlier):** titik yang tidak masuk cluster manapun → label **-1**

**Keunggulan:**
- Tidak perlu menentukan k di awal
- Bisa mendeteksi cluster berbentuk tidak beraturan
- Secara otomatis mengidentifikasi outlier

**Parameter:**
- `eps`: radius neighborhood
- `min_samples`: minimum titik untuk menjadi core point

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

# Dataset berbentuk bulan sabit (non-convex)
X_moons, _ = make_moons(n_samples=300, noise=0.1, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# K-Means pada data non-convex
km_moons = KMeans(n_clusters=2, random_state=0).fit_predict(X_moons)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons, cmap='viridis', s=20)
axes[0].set_title('K-Means (gagal pada bentuk non-convex)')

# DBSCAN pada data non-convex
dbscan = DBSCAN(eps=0.3, min_samples=5)
labels_db = dbscan.fit_predict(X_moons)
colors = ['red' if l == -1 else 'blue' if l == 0 else 'green' for l in labels_db]
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_db, cmap='viridis', s=20)
axes[1].set_title('DBSCAN (berhasil pada bentuk non-convex)')

plt.suptitle('K-Means vs DBSCAN pada Data Non-Convex', fontsize=13)
plt.tight_layout()
plt.show()

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise    = list(labels_db).count(-1)
print(f"DBSCAN: {n_clusters} cluster ditemukan, {n_noise} titik noise/outlier")

## 3.5 Perbandingan Algoritma Clustering

| Algoritma | Perlu tentukan k? | Bisa deteksi outlier? | Bentuk cluster | Kompleksitas |
|---|---|---|---|---|
| **K-Means** |  Ya |  Tidak | Bulat | O(n·k·t) |
| **Agglomerative** |  Ya |  Tidak | Fleksibel | O(n²log n) |
| **DBSCAN** |  Tidak |  Ya | Bebas | O(n log n) |

##  Ringkasan Bab 3

| Konsep | Penjelasan |
|---|---|
| **Scaling** | Normalisasi fitur penting sebelum PCA, clustering, SVM |
| **PCA** | Reduksi dimensi dengan memaksimalkan variansi |
| **NMF** | Faktorisasi non-negatif — cocok untuk gambar/teks |
| **t-SNE** | Visualisasi data berdimensi tinggi |
| **K-Means** | Clustering cepat, perlu tentukan k |
| **Agglomerative** | Hierarchical clustering, bisa dilihat via dendrogram |
| **DBSCAN** | Clustering berbasis densitas, auto-deteksi outlier |

---
 **Next:** Chapter 4 — Representing Data and Engineering Features